In [ ]:
import plotly.express as px
import torch

from aare.constants import TIME
from aare.constants import ANYTIME
from aare.params import read_params
from aare.remote_existenz_store import RemoteExistenzStore

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
# logging.basicConfig(level="DEBUG")

In [ ]:
params = read_params()
store = RemoteExistenzStore()

# Deeper Questions on covariates

After the analysis of the other variables and talking to people, some questions need to be answered.

- Is sunshine duration (smn/ss) a good proxy for global radiation (smn/rad), because there is no radiation forecast?
- Is precipitation (smn/rr) a good proxy for relative humidity (smn/rh), because there is no rh forecast?
- Is the water temperature of the thun lake slow/static enough that it could be used to aid forecasting as past cov (no lake temp forecast)? Spiez (2093) is probably the closest you get to the river temp, but it has no temperature :( Can try Interlaken (2457), which is the river connection between Brienzersee and Thunersee.
- Imputation/Missing Data
  - How many gaps does the air temperature have?
  - Can the air temp be filled the same way as the water temp?
  - THIS SHOULD BE DONE FOR ALL VARIABLES THAT ARE INCLUDED IN MODEL TRAINING


In [ ]:
df = store.query(
    ANYTIME,
    [
        "hydro/temperature:mean_1h@bern",
        "hydro/temperature:mean_1h@thun",
        "hydro/temperature:mean_1h@int",
        "smn/tt:mean_1h@bern",
        "smn/tt:mean_1h@thun",
        "smn/tt:mean_1h@int",
        "smn/ss:sum_1h@bern",
        "smn/rad:sum_1h@bern",
        "smn/rr:sum_1h@bern",
        "smn/rh:sum_1h@bern",
    ],
)
df

## Sunshine Duration as Radiation Proxy

In [ ]:
df_s = df.copy()
df_s[["rad_bern", "ss_bern"]] /= df[["rad_bern", "ss_bern"]].max()
px.scatter(df_s, x=TIME, y=["rad_bern", "ss_bern"]).update_traces(marker={"size": 2})

Visually, there clearly is a high correlation, but it seems that some fine patterns are different like

- some days with low but non-zero sunshine duration have flat 0 radidation
- radiation is often maxed without fluctuations as soon as the sunshine duration approaches ~30min per hour, while sunshine duration has more of a curve most days

In [ ]:
# TODO Gaps in ss. How to imputate?

## Precipitation as Relative Humidity Proxy

In [ ]:
px.scatter(df, x=TIME, y=["rh_bern", "rr_bern"]).update_traces(marker={"size": 2})

## Water temperature BERN, THUN and INT

In [ ]:
px.scatter(df, x=TIME, y=["temperature_bern", "temperature_thun", "temperature_int"]).update_traces(marker={"size": 2})

Starts:

- BERN: 2001
- THUN: 2003
- INT: 2018

INT is much lower (as expected), but not suitable as baseline to just predict the diff to bern on top of (it has even higher variance than bern and thun).

## Air temperature

In [ ]:
px.scatter(df, x=TIME, y=["tt_bern", "tt_thun", "tt_int"]).update_traces(marker={"size": 2})

### Imputation of air temperature